# Graph-Constrained Reasoning (GCR) - Colab Demo

Official implementation of "[Graph-constrained Reasoning: Faithful Reasoning on Knowledge Graphs with Large Language Models](https://arxiv.org/abs/2410.13080)" (ICML 2025).

**What is GCR?**

GCR bridges structured knowledge in Knowledge Graphs (KGs) with unstructured reasoning in LLMs. It ensures **faithful KG-grounded reasoning** by integrating KG structure into the LLM decoding process through a **KG-Trie** — a prefix tree of all valid paths in the KG starting from a question entity. During generation, a `prefix_allowed_tokens_fn` constrains the LLM to only produce tokens that form valid KG paths, achieving **zero reasoning hallucination**.

**Pipeline (2 steps):**
1. **Graph-Constrained Decoding** — A lightweight KG-specialized LLM generates multiple KG-grounded reasoning paths + answer hypotheses using beam search constrained by the KG-Trie.
2. **Graph Inductive Reasoning** — A general LLM (GPT-4, etc.) reasons over the generated paths and hypotheses to produce a final answer.

---

## Setup

In [ ]:
# @title 1. Install dependencies
import sys, os, torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")
print(f"PyTorch: {torch.__version__}")

!pip install -q transformers==4.44 accelerate peft deepspeed tiktoken datasets python-dotenv marisa-trie scikit-learn bitsandbytes trl sentencepiece protobuf wandb networkx

if has_a100:
    print("A100 detected — installing flash-attn-2")
    !pip install -q flash-attn --no-build-isolation

In [ ]:
# @title Clone the repo
import os
if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/rmanluo/graph-constrained-reasoning.git
%cd graph-constrained-reasoning
import sys
sys.path.insert(0, '.')

---
## How GCR Works

### The KG-Trie

The central innovation is a **KG-Trie** — a prefix tree built from all possible paths in the KG starting from a question's topic entity:

```
Question: "Which film directed by Christopher Nolan won Best Picture?"
Topic entity: "Christopher Nolan"

KG paths from Christopher Nolan (2 hops):
  Christopher Nolan -> directed -> Inception -> won_award -> Best_Picture  ✅
  Christopher Nolan -> directed -> Interstellar -> won_award -> Oscar
```

These paths are tokenized and stored in a `MarisaTrie`. During decoding, HuggingFace's `prefix_allowed_tokens_fn` callback filters the logits at every step, allowing only the next token that leads to a valid KG path:

```
Generated so far: "Christopher Nolan -> directed ->"
Trie allows:     ["Inception", "Interstellar", ...]
Trie blocks:     ["is", "was", "a", "the", ... (everything not in KG)]
```

This guarantees every generated reasoning path actually exists in the KG — **zero hallucination on factual paths**.

---
## Step 1: Graph-Constrained Decoding

Run this cell. **Before running, configure your model below.**

In [ ]:
# @title 2. Step 1: Graph-Constrained Decoding
import os, json, sys, torch
from tqdm import tqdm
from datasets import load_dataset
sys.path.insert(0, '.')
from src.llms import get_registed_model
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

# ═══════════════════════════════════════════════════
# MODEL SELECTION
# ═══════════════════════════════════════════════════
#
# Uncomment ONE of the MODEL_PATH lines below.
#
#   A. rmanluo/GCR-Qwen2-0.5B-Instruct     ~1 GB     T4 ✅  A100 ✅
#   B. rmanluo/GCR-Qwen2-1.5B-Instruct     ~3 GB     T4 ✅  A100 ✅
#   C. rmanluo/GCR-Llama-2-7B-chat-hf      ~14 GB    T4 ❌  A100 ✅  (gated)
#   D. rmanluo/GCR-Meta-Llama-3.1-8B-Instruct ~16GB  T4 ❌  A100 ✅  (gated, best)
#
# For gated models (Llama-2, Llama-3.1): you need an HF_TOKEN.
# Uncomment and set os.environ["HF_TOKEN"] below.
# Get yours at: https://huggingface.co/settings/tokens

MODEL_PATH = "rmanluo/GCR-Qwen2-0.5B-Instruct"
# MODEL_PATH = "rmanluo/GCR-Qwen2-1.5B-Instruct"
# MODEL_PATH = "rmanluo/GCR-Llama-2-7B-chat-hf"
# MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"

# Uncomment and fill in if using a gated model:
# os.environ["HF_TOKEN"] = "hf_your_token_here"

# ═══════════════════════════════════════════════════
# OTHER SETTINGS
# ═══════════════════════════════════════════════════
MODEL_NAME = "gcr"                     # keep this as-is
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"                         # "test" or "test[:10]" for quick run
INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
NUM_EXAMPLES = 3

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
ATTN_IMPL = "flash_attention_2" if "A100" in gpu_name else "sdpa"

print(f"Model: {MODEL_PATH}")
print(f"Attention: {ATTN_IMPL}")
print(f"Dataset: {DATASET}")
print(f"Split: {SPLIT}")

import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--data_path', type=str, default=DATA_PATH)
parser.add_argument('--d', '-d', type=str, default=DATASET)
parser.add_argument('--split', type=str, default=SPLIT)
parser.add_argument('--index_path_length', type=int, default=INDEX_LEN)
parser.add_argument('--predict_path', type=str, default='results/GenPaths')
parser.add_argument('--model_name', type=str, default=MODEL_NAME)
parser.add_argument('--force', action='store_true', default=True)
parser.add_argument('--n', type=int, default=1)
parser.add_argument('--undirected', type=lambda x: (str(x).lower() == 'true'), default=False)
parser.add_argument('--debug', action='store_true', default=False)
parser.add_argument('--prompt_mode', type=str, default=PROMPT_MODE)
parser.add_argument('--filter_empty', action='store_true')
parser.add_argument('--add_rule', action='store_true')
parser.add_argument('--prefix', type=str, default='')

LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = 256

print("\nLoading model...")
model = LLM(args)
model.prepare_for_inference()

print("Loading dataset...")
dataset = load_dataset(DATA_PATH + "/" + DATASET, split=SPLIT)
print(f"Loaded {len(dataset)} examples")

input_builder = PathGenerationWithAnswerPromptBuilder(
    model.tokenizer, args.prompt_mode,
    index_path_length=args.index_path_length,
    undirected=args.undirected
)

results = []
for i, data in enumerate(dataset.select(range(NUM_EXAMPLES))):
    qid = data["id"]
    question = data["question"]
    answer = data["answer"]

    print(f"\n{'='*60}")
    print(f"Example {i+1} | ID: {qid}")
    print(f"Question: {question}")
    print(f"Ground truth answer: {answer}")

    input_query, ground_paths, trie = input_builder.process_input(data)
    if trie is None:
        print(">>> No valid KG paths found. Skipping.")
        continue

    start_token_ids = model.tokenizer.convert_tokens_to_ids(
        input_builder.PATH_START_TOKEN
    )
    end_token_ids = model.tokenizer.convert_tokens_to_ids(
        input_builder.PATH_END_TOKEN
    )

    llm_input = model.prepare_model_prompt(input_query)

    print(f"\n>>> Generating {K} reasoning paths via graph-constrained decoding...")
    predictions = model.generate_sentence(
        llm_input, trie,
        start_token_ids=start_token_ids,
        end_token_ids=end_token_ids,
        enable_constrained_by_default=False,
    )

    if predictions is None:
        print(">>> Generation failed.")
        continue

    if isinstance(predictions, str):
        predictions = [predictions]

    print(f"\nGenerated {len(predictions)} path(s):")
    for j, pred in enumerate(predictions):
        display = pred[:200] + "..." if len(pred) > 200 else pred
        print(f"  [{j+1}] {display}")

    results.append({
        "id": qid,
        "question": question,
        "prediction": predictions,
        "ground_truth": answer,
    })

### Understanding the Output

Each prediction contains:
- **KG paths** (e.g., `Christopher Nolan -> directed -> Inception -> won_award -> Best_Picture`)
- **Answer hypothesis** (the entity at the end of the path)

The paths are **guaranteed to exist in the KG** because the trie only allows valid next tokens. This is the "zero hallucination" guarantee.

---
## Step 2: Graph Inductive Reasoning (Optional)

Now we use a general LLM to reason over the multiple paths and produce a final answer.

**Option A:** With an OpenAI key → uses `gpt-4o-mini` (recommended, best results)

**Option B:** Without an OpenAI key → uses a local HuggingFace model (or skip this step)

Set your key in the cell below if you have one.

In [ ]:
# @title 3a. Configure inductive reasoning model
import os

# Set your OpenAI key here if you have one:
os.environ["OPENAI_API_KEY"] = ""  # <-- PASTE YOUR KEY HERE, or leave empty

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))

if USE_OPENAI:
    print("Using gpt-4o-mini for inductive reasoning.")
    REASONING_MODEL = "gpt-4o-mini"
    REASONING_MODEL_PATH = "gpt-4o-mini"
else:
    print("No OpenAI key. Step 3b will be skipped.")
    print("To use a local HF model instead, change the cell below.")

In [ ]:
# @title 3b. Run inductive reasoning
if not USE_OPENAI:
    print("Skipping Step 2 — no OpenAI key configured.")
    print("You can get a key at https://platform.openai.com/api-keys")
else:
    from src.llms import get_registed_model
    from src.qa_prompt_builder import PromptBuilder
    from src.utils.graph_utils import build_graph, get_truth_paths
    from src.utils import path_to_string

    dataset_with_paths = dataset.select(range(NUM_EXAMPLES))

    def merge_predictions(sample):
        for r in results:
            if r["id"] == sample["id"]:
                sample["predicted_paths"] = r["prediction"]
                g = build_graph(sample["graph"])
                truth_paths = get_truth_paths(sample["q_entity"], sample["a_entity"], g)
                sample["ground_paths"] = [path_to_string(p) for p in truth_paths]
                return sample
        sample["predicted_paths"] = []
        sample["ground_paths"] = []
        return sample

    dataset_with_paths = dataset_with_paths.map(merge_predictions)

    reas_LLM = get_registed_model("gpt")

    import argparse
    reas_parser = argparse.ArgumentParser()
    reas_LLM.add_args(reas_parser)
    reas_args = reas_parser.parse_args([])
    reas_args.model_name = REASONING_MODEL
    reas_args.model_path = REASONING_MODEL_PATH
    reas_args.add_path = True

    reas_model = reas_LLM(reas_args)
    reas_model.prepare_for_inference()

    input_builder = PromptBuilder(
        add_path=True, use_true=False,
        maximun_token=reas_model.maximun_token,
        tokenize=reas_model.token_len,
    )

    for data in dataset_with_paths:
        inp = input_builder.process_input(data)
        inp = reas_model.prepare_model_prompt(inp)
        print(f"\n{'='*60}")
        print(f"Question: {data['question']}")
        print(f"\nGround truth: {data['answer']}")
        pred = reas_model.generate_sentence(inp)
        print(f"Final prediction: {pred}")

---
## Summary: The GCR Architecture

```
Question + Topic Entity
         |
         v
+---------------------+    +------------------+
|    Knowledge Graph   |    |  KG-Specialized  |
|   (all triples)      |--->|  LLM (fine-tuned)|
+---------------------+    +-------+----------+
         |                         |
         v                         | prefix_allowed_tokens_fn
+---------------------+            |
|      KG-Trie        |<-----------+|
|  (valid token mask) |
+---------------------+
         |
         v
+---------------------------------------------+
|  Generated: KG paths + Answer hypotheses    |
|  (Guaranteed to exist in KG)                |
+---------------------------------------------+
         |
         v
+---------------------+
|   General LLM        |
|   (inductive reason) |--> Final Answer
+---------------------+
```

### Pre-trained models:
- `rmanluo/GCR-Qwen2-0.5B-Instruct` (smallest, ~1GB)
- `rmanluo/GCR-Qwen2-1.5B-Instruct`
- `rmanluo/GCR-Llama-2-7B-chat-hf`
- `rmanluo/GCR-Meta-Llama-3.1-8B-Instruct` (best)

[Paper](https://arxiv.org/abs/2410.13080) | [GitHub](https://github.com/rmanluo/graph-constrained-reasoning) | [HF Collection](https://huggingface.co/collections/rmanluo/graph-constrained-reasoning-671052e5c808aa5e8c57501a)